# Strategy B — Data preparation with augmentation

**Goal:** produce `strategy_B_with_aug/artifacts/splits.npz` with the same schema as Strategy A, but where the training set includes augmented data while val/test remain 100% real.

**Leakage rule:** an augmented file's split is determined by its **source file's collector**. Augmented files whose source collector is in val/test are dropped — they cannot be used because evaluating on them, or training on them while testing on the source, would create leakage.

**Apples-to-apples with Strategy A:** val and test are identical to A's at the segment level (same files, same segments, same aggregated labels). Only the train set and the `StandardScaler` differ.

In [1]:
# ============================================================================
# CELL 1 — Setup
# ============================================================================
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'config.json').is_file() and (p / 'data').is_dir():
            return p
    raise RuntimeError("project root (config.json + data/) not found above cwd")

ROOT = find_root()
with open(ROOT / 'config.json') as f:
    CONFIG = json.load(f)
P = CONFIG['paths']

print(f"ROOT            = {ROOT}")
print(f"originals (npz) = {ROOT / P['orig_npz']}")
print(f"augmented (npz) = {ROOT / P['aug_npz']}")
print(f"strategy A out  = {ROOT / P['strategy_a_artifacts']}")
print(f"strategy B out  = {ROOT / P['strategy_b_artifacts']}")

ROOT            = /Users/amrabdullah/AI_2026ss/mlpc
originals (npz) = /Users/amrabdullah/AI_2026ss/mlpc/data/audio_features
augmented (npz) = /Users/amrabdullah/AI_2026ss/mlpc/data/audio_features_aug
strategy A out  = /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts
strategy B out  = /Users/amrabdullah/AI_2026ss/mlpc/strategy_B_with_aug/artifacts


## Cell 2 — Inherit Strategy A's partition + load augmentation manifest

Three lookups get built:
- `collector_to_split`  — from A's `collector_partition.csv`
- `file_to_collector`   — from `metadata.csv`
- the augmentation manifest — to know each aug file's source

The manifest's column names are **printed** so we can verify they match what Cell 3 expects. If columns are named differently in your manifest, edit Cell 3's strings accordingly.

In [2]:
# ============================================================================
# CELL 2 — Load A's partition + augmentation manifest
# ============================================================================

# --- 2.1  Strategy A's collector → split mapping --------------------------
collector_partition = pd.read_csv(
    ROOT / P['strategy_a_artifacts'] / 'collector_partition.csv'
)
collector_to_split = dict(
    zip(collector_partition['collector_id'], collector_partition['split'])
)
print(f"Strategy A collector partition:")
print(f"  {collector_partition['split'].value_counts().to_dict()}")
print(f"  total: {len(collector_partition)} collectors\n")

# --- 2.2  metadata: file_id → collector_id --------------------------------
metadata = pd.read_csv(ROOT / P['metadata_csv'])
metadata['file_id'] = metadata['filename'].str.replace(r'\.[^.]+$', '', regex=True)
file_to_collector = dict(zip(metadata['file_id'], metadata['collector_id']))
print(f"metadata: {len(metadata)} files, "
      f"{metadata['collector_id'].nunique()} collectors\n")

# --- 2.3  Augmentation manifest -------------------------------------------
manifest = pd.read_csv(ROOT / P['aug_artifacts'] / 'augmentation_manifest.csv')
print(f"Augmentation manifest: {len(manifest)} rows")
print(f"Columns: {list(manifest.columns)}\n")
print("First 3 rows:")
print(manifest.head(3).to_string())

Strategy A collector partition:
  {'train': 244, 'test': 82, 'val': 82}
  total: 408 collectors

metadata: 3656 files, 408 collectors

Augmentation manifest: 5205 rows
Columns: ['aug_filename', 'cls', 'src_filename', 'src_t_start', 'src_t_end', 'src_onset_s', 'src_offset_s', 'src_duration_s', 'aug_duration_s', 'time_stretch_ratio', 'applied_transforms', 'aug_seed']

First 3 rows:
  aug_filename              cls  src_filename  src_t_start  src_t_end  src_onset_s  src_offset_s  src_duration_s  aug_duration_s  time_stretch_ratio             applied_transforms   aug_seed
0   aug_001090  door_open_close             2            2          9          1.0           5.5             4.5             4.5                 1.0     ["Gain", "AddGaussianSNR"]  468862353
1   aug_001216  door_open_close             4           28         34         14.0          18.0             4.0             4.0                 1.0    ["Shift", "AddGaussianSNR"]  955080931
2   aug_001219  door_open_close             

In [3]:
# Diagnostic — verify the actual type and format of src_filename values
print(f"dtype:           {manifest['src_filename'].dtype}")
print(f"first 5 values:  {manifest['src_filename'].head().tolist()}")
print(f"sample type:     {type(manifest['src_filename'].iloc[0]).__name__}")

# What do the corresponding file_ids in metadata look like?
print(f"\nmetadata file_id sample: {metadata['file_id'].head(3).tolist()}")

# Try a join attempt to see if anything matches as-is
sample_src = manifest['src_filename'].iloc[0]
print(f"\nlookup test: file_to_collector.get({sample_src!r}) = "
      f"{file_to_collector.get(sample_src)}")

dtype:           int64
first 5 values:  [2, 4, 4, 4, 5]
sample type:     int64

metadata file_id sample: ['003075', '001117', '001982']

lookup test: file_to_collector.get(np.int64(2)) = None


## Cell 3 — Apply the leakage rule

**Rule:** aug-source-in-train → augmented file joins training. aug-source-in-val or aug-source-in-test → discarded.

**Why discard rather than redistribute?**

An augmented file *is* the source signal under a small perturbation. Putting it anywhere except where the source lives creates two failure modes:

- **aug-in-val:** the model is trained on the real source and evaluated on a perturbed version of it. The reported metric becomes "invariance to our chosen transforms," not "generalization to new homes."
- **aug-in-test:** identical problem with extra reporting bias.

Even leaving real val/test segments untouched while *adding* aug-source-in-val to val would just inflate val with synthetic copies of val signal — useless.

**Cost:** ~40% of aug files are likely discarded (since ~40% of source collectors live in val+test). This is the price of measuring honestly. The actual number prints below.

> **Adjust here if manifest columns differ.** The two strings `'aug_filename'` and `'src_filename'` must match your manifest's actual columns (printed in Cell 2's output).

In [4]:
# ============================================================================
# CELL 3 — Apply leakage rule: drop aug whose source is in val/test
# ============================================================================
# Manifest column names — UPDATE if Cell 2's printed columns differ:
COL_AUG_FN = 'aug_filename'
COL_SRC_FN = 'src_filename'

assert COL_AUG_FN in manifest.columns, f"manifest is missing '{COL_AUG_FN}'"
assert COL_SRC_FN in manifest.columns, f"manifest is missing '{COL_SRC_FN}'"

# --- 3.1  Strip extensions to get file_ids --------------------------------
manifest['aug_file_id'] = manifest[COL_AUG_FN].str.replace(r'\.[^.]+$', '', regex=True)
# src_filename was stored as int64 in the manifest (zero-padding lost during
# CSV round-trip). Convert back to the canonical 6-digit zero-padded form
# used by metadata.csv and the .npz filenames.
manifest['src_file_id'] = manifest[COL_SRC_FN].astype(int).map(lambda x: f"{x:06d}")

# Verify every src_file_id actually exists in metadata
unknown_src_ids = set(manifest['src_file_id']) - set(metadata['file_id'])
if unknown_src_ids:
    sample = list(unknown_src_ids)[:5]
    raise AssertionError(
        f"{len(unknown_src_ids)} src_file_id values don't appear in metadata.csv. "
        f"Sample: {sample}. The 6-digit zero-padding assumption may be wrong."
    )
print(f"All {len(set(manifest['src_file_id']))} unique src_file_ids resolve to metadata rows.")

# --- 3.2  Map aug → source → collector → split ----------------------------
manifest['src_collector_id'] = manifest['src_file_id'].map(file_to_collector)
manifest['split']            = manifest['src_collector_id'].map(collector_to_split)

# --- 3.3  Diagnostic checks -----------------------------------------------
n_unmapped_collector = manifest['src_collector_id'].isna().sum()
n_unmapped_split     = manifest['split'].isna().sum()
print(f"aug files with unmapped source collector: {n_unmapped_collector}  (expect 0)")
print(f"aug files with unmapped split:            {n_unmapped_split}  (expect 0)")
assert n_unmapped_collector == 0 and n_unmapped_split == 0

# --- 3.4  Distribution across the three splits ----------------------------
print(f"\nAug files per split (before drop):")
split_counts = manifest['split'].value_counts()
for s in ['train', 'val', 'test']:
    print(f"  {s:5s}: {split_counts.get(s, 0):>5d}")
print(f"  total: {len(manifest):>5d}")

# --- 3.5  Drop val/test-source aug files ----------------------------------
n_before        = len(manifest)
manifest_train  = manifest[manifest['split'] == 'train'].copy().reset_index(drop=True)
n_after         = len(manifest_train)
n_dropped       = n_before - n_after

print(f"\nLEAKAGE RULE applied:")
print(f"  before:                          {n_before:>5d}")
print(f"  dropped (source in val/test):    {n_dropped:>5d}  ({n_dropped/n_before:.1%})")
print(f"  remaining (source in train):     {n_after:>5d}  ({n_after/n_before:.1%})")

All 1521 unique src_file_ids resolve to metadata rows.
aug files with unmapped source collector: 0  (expect 0)
aug files with unmapped split:            0  (expect 0)

Aug files per split (before drop):
  train:  3162
  val  :  1016
  test :  1027
  total:  5205

LEAKAGE RULE applied:
  before:                           5205
  dropped (source in val/test):     2043  (39.3%)
  remaining (source in train):      3162  (60.7%)


In [9]:
# ============================================================================
# CELL 3.5 — Per-class effect of the leakage rule (diagnostic)
# ============================================================================
per_class = manifest.groupby(['cls', 'split']).size().unstack(fill_value=0)
per_class['total']    = per_class.sum(axis=1)
per_class['kept']     = per_class.get('train', 0)
per_class['dropped']  = per_class['total'] - per_class['kept']
per_class['kept_pct'] = (per_class['kept'] / per_class['total'] * 100).round(1)
per_class = per_class.sort_values('total', ascending=False)

# Force the count columns to int (unstack with fill_value=0 promoted them to float)
for col in ['total', 'kept', 'dropped']:
    per_class[col] = per_class[col].astype(int)

print("Augmentation effect per class after leakage rule:\n")
print(f"  {'class':<30s}  {'total':>6s}  {'kept':>6s}  {'dropped':>7s}  {'kept_%':>6s}")
print("  " + "-" * 64)
for c, row in per_class.iterrows():
    print(f"  {c:<30s}  {int(row['total']):>6d}  {int(row['kept']):>6d}  "
          f"{int(row['dropped']):>7d}  {row['kept_pct']:>5.1f}%")

print(f"\n  range of kept_%: {per_class['kept_pct'].min():.1f}% – {per_class['kept_pct'].max():.1f}%")
print(f"  tight range (~55-65%): uniform drop, proceed")
print(f"  wide range (e.g. 40-80%): some rare classes hit disproportionately")

Augmentation effect per class after leakage rule:

  class                            total    kept  dropped  kept_%
  ----------------------------------------------------------------
  window_open_close                 1448     876      572   60.5%
  light_switch                      1202     738      464   61.4%
  wardrobe_drawer_open_close        1086     676      410   62.2%
  bell_ringing                       933     542      391   58.1%
  door_open_close                    221     140       81   63.3%
  toilet_flushing                    214     129       85   60.3%
  coffee_machine                     101      61       40   60.4%

  range of kept_%: 58.1% – 63.3%
  tight range (~55-65%): uniform drop, proceed
  wide range (e.g. 40-80%): some rare classes hit disproportionately


In [10]:
# Save the per-class table for the report
per_class.to_csv(ROOT / P['strategy_b_artifacts'] / 'augmentation_leakage_table.csv')
print(f"Saved: {ROOT / P['strategy_b_artifacts'] / 'augmentation_leakage_table.csv'}")

Saved: /Users/amrabdullah/AI_2026ss/mlpc/strategy_B_with_aug/artifacts/augmentation_leakage_table.csv


## Cell 4 — Shared helpers

These must match Strategy A's preparation exactly so that B's real segments match A's segment-by-segment before scaling. The four pieces:

- `FEATURE_KEYS` — 56 named groups → 960 dims after expansion.
- `BASE_DIMS` — feature group → base dimensionality (e.g. `melspect: 128`, `contrast: 7`).
- `aggregate_labels` — per-annotator binarization at τ=0.5, then strict majority vote.
- `build_feature_matrix` — concatenate the 56 keys along axis 1 to produce `[T, 960]`.

For A=1 the majority rule reduces to "any annotator"; for A=2 it becomes unanimous; for A=3 it requires at least 2.

In [11]:
# ============================================================================
# CELL 4 — Shared helpers
# ============================================================================

FEATURE_KEYS = [
    "zcr_mean",         "zcr_std",         "zcr_min",         "zcr_max",
    "melspect_mean",    "melspect_std",    "melspect_min",    "melspect_max",
    "mfcc_mean",        "mfcc_std",        "mfcc_min",        "mfcc_max",
    "mfcc_d_mean",      "mfcc_d_std",      "mfcc_d_min",      "mfcc_d_max",
    "mfcc_d2_mean",     "mfcc_d2_std",     "mfcc_d2_min",     "mfcc_d2_max",
    "flux_mean",        "flux_std",        "flux_min",        "flux_max",
    "flatness_mean",    "flatness_std",    "flatness_min",    "flatness_max",
    "centroid_mean",    "centroid_std",    "centroid_min",    "centroid_max",
    "bandwidth_mean",   "bandwidth_std",   "bandwidth_min",   "bandwidth_max",
    "contrast_mean",    "contrast_std",    "contrast_min",    "contrast_max",
    "rolloff_low_mean", "rolloff_low_std", "rolloff_low_min", "rolloff_low_max",
    "rolloff_high_mean","rolloff_high_std","rolloff_high_min","rolloff_high_max",
    "energy_mean",      "energy_std",      "energy_min",      "energy_max",
    "power_mean",       "power_std",       "power_min",       "power_max",
]

BASE_DIMS = {
    "zcr": 1, "melspect": 128, "mfcc": 32, "mfcc_d": 32, "mfcc_d2": 32,
    "flux": 1, "flatness": 1, "centroid": 1, "bandwidth": 1, "contrast": 7,
    "rolloff_low": 1, "rolloff_high": 1, "energy": 1, "power": 1,
}

HEAVY_TAILED       = ["energy", "power"]   # log1p before scaling
OVERLAP_THRESHOLD  = 0.5                   # per-annotator binarization


def aggregate_labels(annotations):
    # [T, C, A] float annotations -> [T, C] int8 labels.
    # Per-annotator binarization at tau=0.5, then strict majority vote.
    # A=1 -> any annotator    (votes > 0.5 -> 1)
    # A=2 -> unanimous         (votes > 1)
    # A=3 -> >= 2 of 3         (votes > 1.5)
    T, C, A = annotations.shape
    ann_bin = (annotations > OVERLAP_THRESHOLD).astype(np.int8)   # [T,C,A]
    votes   = ann_bin.sum(axis=2)                                 # [T,C]
    return (votes > A / 2).astype(np.int8)                        # [T,C]


def build_feature_matrix(npz_data, feature_keys=FEATURE_KEYS):
    # Concatenate the 56 named feature arrays along the feature axis -> [T, 960].
    feats = []
    for k in feature_keys:
        arr = npz_data[k]
        if arr.ndim == 1:
            arr = arr[:, None]                  # promote [T] → [T, 1]
        feats.append(arr.astype(np.float32))
    return np.concatenate(feats, axis=1)        # [T, D]


# Sanity check: feature dimensions
expected_D = sum(BASE_DIMS[k.rsplit("_", 1)[0]] for k in FEATURE_KEYS)
print(f"FEATURE_KEYS:    {len(FEATURE_KEYS)} groups")
print(f"Expected D:      {expected_D} (must be 960)")
print(f"HEAVY_TAILED:    {HEAVY_TAILED}")
print(f"OVERLAP_THRESH:  {OVERLAP_THRESHOLD}")
assert expected_D == 960, f"Feature-key accounting mismatch: {expected_D} != 960"

FEATURE_KEYS:    56 groups
Expected D:      960 (must be 960)
HEAVY_TAILED:    ['energy', 'power']
OVERLAP_THRESH:  0.5


## Cell 5 — Load original (real) files

Iteration order matches Strategy A's data prep: `sorted(...glob('*.npz'))`. Per-file we build the `[T, 960]` feature matrix and the `[T, 15]` aggregated labels, tag with the file's collector and split, and store. The full `data` dict is discarded right after extraction to keep memory bounded.

In [12]:
# ============================================================================
# CELL 5 — Load all original files: features + aggregated labels
# ============================================================================
ORIG_NPZ_DIR = ROOT / P['orig_npz']
npz_real     = sorted(ORIG_NPZ_DIR.glob('*.npz'))
print(f"Found {len(npz_real)} original npz files")

real_records = []
t0 = time.time()
for npz_path in tqdm(npz_real, desc='loading real'):
    file_id      = npz_path.stem
    data         = dict(np.load(npz_path, allow_pickle=True))
    X            = build_feature_matrix(data)
    Y            = aggregate_labels(data['annotations'])
    collector_id = file_to_collector[file_id]
    split        = collector_to_split[collector_id]
    real_records.append({
        'file_id':      file_id,
        'collector_id': collector_id,
        'split':        split,
        'T':            X.shape[0],
        'X':            X,
        'Y':            Y,
    })

CLASS_NAMES = list(np.load(npz_real[0], allow_pickle=True)['class_names'])
total_T = sum(r['T'] for r in real_records)
print(f"\nLoaded {len(real_records)} real files in {time.time()-t0:.1f}s")
print(f"Total real segments: {total_T:,}  (expect 168,239)")
assert total_T == 168_239, f"Real-segment total {total_T} ≠ 168,239"
print(f"Classes (canonical order): {CLASS_NAMES}")

Found 3656 original npz files


loading real:   0%|          | 0/3656 [00:00<?, ?it/s]


Loaded 3656 real files in 14.6s
Total real segments: 168,239  (expect 168,239)
Classes (canonical order): ['bell_ringing', 'coffee_machine', 'cutlery_dishes', 'door_open_close', 'footsteps', 'keyboard_typing', 'keychain', 'light_switch', 'microwave', 'phone_ringing', 'running_water', 'toilet_flushing', 'vacuum_cleaner', 'wardrobe_drawer_open_close', 'window_open_close']


## Cell 6 — Load augmented files (train-source only)

We iterate `data/audio_features_aug/*.npz` in sorted order but **keep only those whose `aug_file_id` survived Cell 3's filter** (source collector in train). Same feature/label pipeline as for real files.

In [13]:
# ============================================================================
# CELL 6 — Load augmented files whose source is in train
# ============================================================================
AUG_NPZ_DIR  = ROOT / P['aug_npz']
keep_aug_ids = set(manifest_train['aug_file_id'])
all_aug_npz  = sorted(AUG_NPZ_DIR.glob('*.npz'))
aug_npz_train = [p for p in all_aug_npz if p.stem in keep_aug_ids]

print(f"aug files on disk:       {len(all_aug_npz):>5d}")
print(f"aug files to keep (Cell 3): {len(keep_aug_ids):>5d}")
print(f"matched on disk:         {len(aug_npz_train):>5d}")
missing = len(keep_aug_ids) - len(aug_npz_train)
assert missing == 0, f"{missing} aug files referenced in manifest but missing on disk"

aug_to_src = dict(zip(manifest_train['aug_file_id'], manifest_train['src_file_id']))

aug_records = []
t0 = time.time()
for npz_path in tqdm(aug_npz_train, desc='loading aug'):
    aug_file_id  = npz_path.stem
    src_file_id  = aug_to_src[aug_file_id]
    collector_id = file_to_collector[src_file_id]
    data         = dict(np.load(npz_path, allow_pickle=True))
    X            = build_feature_matrix(data)
    Y            = aggregate_labels(data['annotations'])
    aug_records.append({
        'file_id':      aug_file_id,
        'src_file_id':  src_file_id,
        'collector_id': collector_id,
        'split':        'train',
        'T':            X.shape[0],
        'X':            X,
        'Y':            Y,
    })

total_T_aug = sum(r['T'] for r in aug_records)
print(f"\nLoaded {len(aug_records)} aug files in {time.time()-t0:.1f}s")
print(f"Total aug segments: {total_T_aug:,}")
print(f"Avg segments per aug file: {total_T_aug/max(len(aug_records),1):.1f}")

aug files on disk:        5205
aug files to keep (Cell 3):  3162
matched on disk:          3162


loading aug:   0%|          | 0/3162 [00:00<?, ?it/s]


Loaded 3162 aug files in 12.2s
Total aug segments: 14,360
Avg segments per aug file: 4.5


## Cell 7 — Build combined arrays + partition indices

Concatenation order: **all real first, then all aug.** So real segments live in `[0, N_real)`, aug segments in `[N_real, N_real + N_aug)`. Split indices select from this combined view.

We also keep an auxiliary `source_all` array (`'real'` / `'aug'`) per segment so we can verify the leakage rule held *and* save it to the output for downstream ablation.

In [14]:
# ============================================================================
# CELL 7 — Combined arrays + partition indices
# ============================================================================

# --- 7.1  Concatenate real ------------------------------------------------
X_real         = np.concatenate([r['X'] for r in real_records], axis=0)
Y_real         = np.concatenate([r['Y'] for r in real_records], axis=0)
split_real     = np.concatenate([np.array([r['split']] * r['T']) for r in real_records])
collector_real = np.concatenate([np.array([r['collector_id']] * r['T']) for r in real_records])

# --- 7.2  Concatenate aug -------------------------------------------------
if len(aug_records) > 0:
    X_aug         = np.concatenate([r['X'] for r in aug_records], axis=0)
    Y_aug         = np.concatenate([r['Y'] for r in aug_records], axis=0)
    split_aug     = np.array(['train'] * X_aug.shape[0])
    collector_aug = np.concatenate(
        [np.array([r['collector_id']] * r['T']) for r in aug_records]
    )
else:
    X_aug = np.empty((0, X_real.shape[1]), dtype=np.float32)
    Y_aug = np.empty((0, Y_real.shape[1]), dtype=np.int8)
    split_aug     = np.empty((0,), dtype=split_real.dtype)
    collector_aug = np.empty((0,), dtype=collector_real.dtype)

N_real = X_real.shape[0]
N_aug  = X_aug.shape[0]

# --- 7.3  Stack into the combined feature space ---------------------------
X_all         = np.concatenate([X_real, X_aug], axis=0)
Y_all         = np.concatenate([Y_real, Y_aug], axis=0)
split_all     = np.concatenate([split_real, split_aug])
collector_all = np.concatenate([collector_real, collector_aug])
source_all    = np.array(['real'] * N_real + ['aug'] * N_aug)

# Free per-record duplicates
del real_records, aug_records, X_real, X_aug, Y_real, Y_aug
del split_real, split_aug, collector_real, collector_aug

# --- 7.4  Partition indices -----------------------------------------------
train_idx = np.where(split_all == 'train')[0]
val_idx   = np.where(split_all == 'val')[0]
test_idx  = np.where(split_all == 'test')[0]

print(f"=== COMBINED ARRAYS ===")
print(f"  N_real:  {N_real:>7,d}")
print(f"  N_aug:   {N_aug:>7,d}")
print(f"  N_total: {N_real + N_aug:>7,d}\n")

print(f"=== PARTITION ===")
def _summary(name, idx):
    n_r = int((source_all[idx] == 'real').sum())
    n_a = int((source_all[idx] == 'aug').sum())
    print(f"  {name:5s}: {len(idx):>7,d}  (real: {n_r:>7,d}  aug: {n_a:>7,d})")
_summary('train', train_idx)
_summary('val',   val_idx)
_summary('test',  test_idx)

# --- 7.5  Leakage assertions ----------------------------------------------
assert (source_all[val_idx]  == 'aug').sum() == 0, "LEAK: aug in val"
assert (source_all[test_idx] == 'aug').sum() == 0, "LEAK: aug in test"
print(f"\n  PASS  no augmented segments in val or test")

=== COMBINED ARRAYS ===
  N_real:  168,239
  N_aug:    14,360
  N_total: 182,599

=== PARTITION ===
  train: 114,526  (real: 100,166  aug:  14,360)
  val  :  34,245  (real:  34,245  aug:       0)
  test :  33,828  (real:  33,828  aug:       0)

  PASS  no augmented segments in val or test


## Cell 8 — Re-verify zero collector overlap

The leakage rule should preserve A's group-aware property. Quick set-intersection check.

In [15]:
# ============================================================================
# CELL 8 — Verify zero collector overlap (sanity check)
# ============================================================================
train_collectors_B = set(collector_all[train_idx])
val_collectors_B   = set(collector_all[val_idx])
test_collectors_B  = set(collector_all[test_idx])

overlap_tv = train_collectors_B & val_collectors_B
overlap_tt = train_collectors_B & test_collectors_B
overlap_vt = val_collectors_B   & test_collectors_B

print(f"train ∩ val   = {len(overlap_tv)}  (expect 0)")
print(f"train ∩ test  = {len(overlap_tt)}  (expect 0)")
print(f"val   ∩ test  = {len(overlap_vt)}  (expect 0)")
assert not (overlap_tv or overlap_tt or overlap_vt), "COLLECTOR LEAKAGE"

print(f"\nCollectors per split:")
print(f"  train: {len(train_collectors_B)}  (expect 244)")
print(f"  val:   {len(val_collectors_B)}    (expect 82)")
print(f"  test:  {len(test_collectors_B)}   (expect 82)")
print(f"\n  PASS  zero collector overlap")

train ∩ val   = 0  (expect 0)
train ∩ test  = 0  (expect 0)
val   ∩ test  = 0  (expect 0)

Collectors per split:
  train: 244  (expect 244)
  val:   82    (expect 82)
  test:  82   (expect 82)

  PASS  zero collector overlap


## Cell 9 — log1p compression on heavy-tailed columns

Same recipe as Strategy A. `energy` and `power` features have extreme outliers (post-scaling max was ~80σ until we added this step) — `log1p` brings them down to ~22σ. The mask is computed deterministically from `BASE_DIMS`, not picked by hand.

Important: log1p is **elementwise** and depends only on each value, so it does not leak across the partition.

In [16]:
# ============================================================================
# CELL 9 — log1p on energy/power columns
# ============================================================================
D = X_all.shape[1]
log_mask = np.zeros(D, dtype=bool)
col = 0
for k in FEATURE_KEYS:
    base    = k.rsplit("_", 1)[0]
    arr_dim = BASE_DIMS[base]
    if base in HEAVY_TAILED:
        log_mask[col:col + arr_dim] = True
    col += arr_dim
assert col == D, f"Column accounting mismatch: {col} != {D}"

print(f"Log-transforming {log_mask.sum()} columns (energy/power); D = {D}")

X_all_log = X_all.copy()
X_all_log[:, log_mask] = np.log1p(np.clip(X_all_log[:, log_mask], 0, None))

# Show the effect on affected cols only
hi_before = X_all[:, log_mask]
hi_after  = X_all_log[:, log_mask]
print(f"\nHeavy-tailed BEFORE log1p:  min={hi_before.min():.2f}  max={hi_before.max():.2f}  mean={hi_before.mean():.2f}")
print(f"Heavy-tailed AFTER  log1p:  min={hi_after.min():.2f}  max={hi_after.max():.2f}  mean={hi_after.mean():.2f}")

del X_all   # free

Log-transforming 8 columns (energy/power); D = 960

Heavy-tailed BEFORE log1p:  min=0.00  max=67940.88  mean=116.79
Heavy-tailed AFTER  log1p:  min=0.00  max=11.13  mean=2.56


## Cell 10 — Fit `StandardScaler` on train, transform all three splits

Scaler is fit on train **only** (real + aug). This is the right thing: aug samples *are* training data by construction. The scaler differs slightly from A's because the train distribution differs slightly (more positives for rare classes, same overall shape).

In [17]:
# ============================================================================
# CELL 10 — Fit scaler on train, transform val/test
# ============================================================================
X_train = X_all_log[train_idx]
X_val   = X_all_log[val_idx]
X_test  = X_all_log[test_idx]
Y_train = Y_all[train_idx]
Y_val   = Y_all[val_idx]
Y_test  = Y_all[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled   = scaler.transform(X_val).astype(np.float32)
X_test_scaled  = scaler.transform(X_test).astype(np.float32)

print(f"Pre-scaling   X_train: min={X_train.min():.2f}  max={X_train.max():.2f}  "
      f"mean={X_train.mean():.2f}  std={X_train.std():.2f}")
print(f"Post-scaling  X_train: min={X_train_scaled.min():.2f}  max={X_train_scaled.max():.2f}  "
      f"mean={X_train_scaled.mean():.4f}  std={X_train_scaled.std():.4f}")

col_max_abs = np.abs(X_train_scaled).max(axis=0)
print(f"\nPer-column |z| max  —  median: {np.median(col_max_abs):.2f}, "
      f"95th: {np.percentile(col_max_abs, 95):.2f}, "
      f"max: {col_max_abs.max():.2f}")

near_const = np.where(scaler.scale_ < 1e-8)[0]
print(f"Near-constant features on train (zero variance): {len(near_const)}")

del X_all_log, X_train, X_val, X_test   # free unscaled copies

Pre-scaling   X_train: min=-130.25  max=8000.00  mean=36.48  std=368.57
Post-scaling  X_train: min=-18.47  max=21.42  mean=0.0000  std=1.0000

Per-column |z| max  —  median: 6.99, 95th: 14.17, max: 21.42
Near-constant features on train (zero variance): 0


## Cell 11 — Compare per-class distributions to Strategy A

Two checks:

1. **Val and test rates MUST be identical to A's** — same segments, same aggregation. If they differ, something's wrong with our label code.
2. **Train rates SHOULD differ** — that's literally what augmentation is for. Rare classes get bumped, common classes stay roughly where they were.

The table prints both side by side.

In [18]:
# ============================================================================
# CELL 11 — Per-class positive rate per split, B vs A
# ============================================================================
splits_A = dict(np.load(
    ROOT / P['strategy_a_artifacts'] / 'splits.npz', allow_pickle=True
))

p_tr_A = splits_A['Y_train'].mean(axis=0)
p_va_A = splits_A['Y_val'].mean(axis=0)
p_te_A = splits_A['Y_test'].mean(axis=0)

p_tr_B = Y_train.mean(axis=0)
p_va_B = Y_val.mean(axis=0)
p_te_B = Y_test.mean(axis=0)

print(f"=== Per-class positive rate (%) — Strategy A vs B ===\n")
header = f"  {'class':<30s}  {'A_tr':>6s}  {'B_tr':>6s}  {'Δtr':>6s}  | {'A_val':>6s}  {'B_val':>6s}  | {'A_te':>6s}  {'B_te':>6s}"
print(header)
print("  " + "-" * (len(header) - 2))
for i, c in enumerate(CLASS_NAMES):
    d_tr = (p_tr_B[i] - p_tr_A[i]) * 100
    print(f"  {c:<30s}  "
          f"{p_tr_A[i]*100:>5.2f}%  {p_tr_B[i]*100:>5.2f}%  {d_tr:>+5.2f}%  | "
          f"{p_va_A[i]*100:>5.2f}%  {p_va_B[i]*100:>5.2f}%  | "
          f"{p_te_A[i]*100:>5.2f}%  {p_te_B[i]*100:>5.2f}%")

# Val and test must be identical
assert np.allclose(p_va_A, p_va_B, atol=1e-6), "Val rates differ from A (should be identical)"
assert np.allclose(p_te_A, p_te_B, atol=1e-6), "Test rates differ from A (should be identical)"
print(f"\n  PASS  val and test rates match A exactly (same segments, same aggregation)")
print(f"  Train rates differ — that's the augmentation effect.")

=== Per-class positive rate (%) — Strategy A vs B ===

  class                             A_tr    B_tr     Δtr  |  A_val   B_val  |   A_te    B_te
  ------------------------------------------------------------------------------------------
  bell_ringing                     1.09%   3.31%  +2.23%  |  1.40%   1.40%  |  1.43%   1.43%
  coffee_machine                   3.78%   4.57%  +0.79%  |  3.13%   3.13%  |  3.65%   3.65%
  cutlery_dishes                   6.03%   5.32%  -0.71%  |  5.34%   5.34%  |  6.05%   6.05%
  door_open_close                  3.57%   3.58%  +0.01%  |  3.44%   3.44%  |  3.18%   3.18%
  footsteps                       11.41%  10.32%  -1.09%  | 11.78%  11.78%  | 12.58%  12.58%
  keyboard_typing                  9.11%   8.07%  -1.04%  |  9.07%   9.07%  |  9.85%   9.85%
  keychain                         4.56%   4.08%  -0.48%  |  4.53%   4.53%  |  4.36%   4.36%
  light_switch                     0.27%   0.89%  +0.62%  |  0.41%   0.41%  |  0.26%   0.26%
  microwave    

In [19]:
import pandas as pd
report_df = pd.DataFrame({
    'class':  CLASS_NAMES,
    'A_train': p_tr_A, 'B_train': p_tr_B, 'delta_train': p_tr_B - p_tr_A,
    'A_val':   p_va_A, 'B_val':   p_va_B,
    'A_test':  p_te_A, 'B_test':  p_te_B,
})
report_df.to_csv(ROOT / P['strategy_b_artifacts'] / 'per_class_rates_A_vs_B.csv', index=False)
print(f"Saved per-class rates table")

Saved per-class rates table


## Cell 12 — Save `splits.npz` (schema parity with A + extras)

Schema parity (so downstream training code is identical):

```
X_train, X_val, X_test          float32  scaled features
Y_train, Y_val, Y_test          int8     multi-label
feature_names                   <U18     56 group names
class_names                     <U26     15 class names
train_idx, val_idx, test_idx    int64    indices into B's X_all
scaler_mean, scaler_scale       float64  scaler params
```

Strategy B extras (for ablation / traceability):

```
source_train, source_val, source_test   <U4   'real' or 'aug' per segment
n_real_total, n_aug_total               int   counts for the report
```

In [20]:
# ============================================================================
# CELL 12 — Save splits.npz (schema parity with A + extras)
# ============================================================================
feature_names = np.array(FEATURE_KEYS, dtype='<U18')

OUT_PATH = ROOT / P['strategy_b_artifacts'] / 'splits.npz'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    OUT_PATH,
    # --- Schema parity with Strategy A ---
    X_train       = X_train_scaled,
    X_val         = X_val_scaled,
    X_test        = X_test_scaled,
    Y_train       = Y_train.astype(np.int8),
    Y_val         = Y_val.astype(np.int8),
    Y_test        = Y_test.astype(np.int8),
    feature_names = feature_names,
    class_names   = np.array(CLASS_NAMES),
    train_idx     = train_idx.astype(np.int64),
    val_idx       = val_idx.astype(np.int64),
    test_idx      = test_idx.astype(np.int64),
    scaler_mean   = scaler.mean_.astype(np.float64),
    scaler_scale  = scaler.scale_.astype(np.float64),
    # --- Strategy B extras ---
    source_train  = source_all[train_idx],
    source_val    = source_all[val_idx],
    source_test   = source_all[test_idx],
    n_real_total  = np.int64(N_real),
    n_aug_total   = np.int64(N_aug),
)

print(f"Saved: {OUT_PATH}")
print(f"  size: {OUT_PATH.stat().st_size / 1e6:.1f} MB")

# Reload sanity check
verify = dict(np.load(OUT_PATH, allow_pickle=True))
print(f"\nKeys in saved file ({len(verify)}):")
for k, v in sorted(verify.items()):
    shape = v.shape if hasattr(v, 'shape') else 'scalar'
    print(f"  {k:18s}  shape={shape}  dtype={v.dtype}")

Saved: /Users/amrabdullah/AI_2026ss/mlpc/strategy_B_with_aug/artifacts/splits.npz
  size: 580.6 MB

Keys in saved file (18):
  X_test              shape=(33828, 960)  dtype=float32
  X_train             shape=(114526, 960)  dtype=float32
  X_val               shape=(34245, 960)  dtype=float32
  Y_test              shape=(33828, 15)  dtype=int8
  Y_train             shape=(114526, 15)  dtype=int8
  Y_val               shape=(34245, 15)  dtype=int8
  class_names         shape=(15,)  dtype=<U26
  feature_names       shape=(56,)  dtype=<U18
  n_aug_total         shape=()  dtype=int64
  n_real_total        shape=()  dtype=int64
  scaler_mean         shape=(960,)  dtype=float64
  scaler_scale        shape=(960,)  dtype=float64
  source_test         shape=(33828,)  dtype=<U4
  source_train        shape=(114526,)  dtype=<U4
  source_val          shape=(34245,)  dtype=<U4
  test_idx            shape=(33828,)  dtype=int64
  train_idx           shape=(114526,)  dtype=int64
  val_idx             s

## Cell 13 — Summary for the report

In [21]:
# ============================================================================
# CELL 13 — Summary
# ============================================================================
n_aug_train  = int((source_all[train_idx] == 'aug').sum())
n_real_train = int((source_all[train_idx] == 'real').sum())

print("=" * 70)
print("STRATEGY B SUMMARY")
print("=" * 70)

print(f"\nInherited from Strategy A:")
print(f"  collector partition:           244 / 82 / 82  (train / val / test)")
print(f"  zero collector overlap:        verified")

print(f"\nAugmentation manifest:")
print(f"  total aug files generated:     {n_before:>7,d}")
print(f"  dropped (source in val/test):  {n_dropped:>7,d}  ({n_dropped/n_before:>5.1%})")
print(f"  used (source in train):        {n_after:>7,d}  ({n_after/n_before:>5.1%})")

print(f"\nFinal partition (segments):")
print(f"  train:  {len(train_idx):>7,d}  (real: {n_real_train:>7,d}  +  aug: {n_aug_train:>7,d})")
print(f"          growth vs A's train:   +{n_aug_train:,} segments "
      f"({n_aug_train/n_real_train:+.1%})")
print(f"  val:    {len(val_idx):>7,d}  (real: {len(val_idx):>7,d}, aug: 0)  — identical to A")
print(f"  test:   {len(test_idx):>7,d}  (real: {len(test_idx):>7,d}, aug: 0)  — identical to A")

print(f"\nScaler:")
print(f"  fit on B's train (real + aug)")
print(f"  scaler_mean[0]:  A={float(splits_A['scaler_mean'][0]):>8.4f}   B={float(scaler.mean_[0]):>8.4f}")
print(f"  scaler_scale[0]: A={float(splits_A['scaler_scale'][0]):>8.4f}   B={float(scaler.scale_[0]):>8.4f}")

print(f"\nOutput: {OUT_PATH}")
print(f"  schema parity with A:  ✓")
print(f"  B extras:              source_train/val/test, n_real_total, n_aug_total")

STRATEGY B SUMMARY

Inherited from Strategy A:
  collector partition:           244 / 82 / 82  (train / val / test)
  zero collector overlap:        verified

Augmentation manifest:
  total aug files generated:       5,205
  dropped (source in val/test):    2,043  (39.3%)
  used (source in train):          3,162  (60.7%)

Final partition (segments):
  train:  114,526  (real: 100,166  +  aug:  14,360)
          growth vs A's train:   +14,360 segments (+14.3%)
  val:     34,245  (real:  34,245, aug: 0)  — identical to A
  test:    33,828  (real:  33,828, aug: 0)  — identical to A

Scaler:
  fit on B's train (real + aug)
  scaler_mean[0]:  A=  0.1890   B=  0.1878
  scaler_scale[0]: A=  0.1299   B=  0.1274

Output: /Users/amrabdullah/AI_2026ss/mlpc/strategy_B_with_aug/artifacts/splits.npz
  schema parity with A:  ✓
  B extras:              source_train/val/test, n_real_total, n_aug_total
